In [0]:
df = spark.table("workspace.default.model_data_2015_2022")

print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")

Rows: 397,423
Columns: 31


In [0]:
train_df = df.filter(df.origination_vintage <= 2019)
val_df = df.filter(df.origination_vintage == 2020)

print(f"Training mortgages: {train_df.count():,}")
print(f"Validation mortgages: {val_df.count():,}")

Training mortgages: 247,885
Validation mortgages: 49,877


In [0]:
numeric_features = [
    "credit_score",
    "dti",
    "cltv",
    "original_upb",
    "ltv",
    "original_interest_rate",
    "original_loan_term",
    "mi_percent",
    "num_units",
    "num_borrowers"
]

categorical_features = [
    "first_time_homebuyer",
    "occupancy_status",
    "channel",
    "property_type",
    "loan_purpose",
    "super_conforming_flag"
]

target = "seriously_delinquent_24m"

model_columns = numeric_features + categorical_features + [target]

train_pd = train_df.select(model_columns).toPandas()
val_pd = val_df.select(model_columns).toPandas()

print(train_pd.shape)
print(val_pd.shape)

(247885, 17)
(49877, 17)


In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

print("Pipeline created.")

Pipeline created.


In [0]:
X_train = train_pd[numeric_features + categorical_features]
y_train = train_pd[target].astype(int)

X_val = val_pd[numeric_features + categorical_features]
y_val = val_pd[target].astype(int)

model.fit(X_train, y_train)

print("Model trained.")

Model trained.


In [0]:
from sklearn.metrics import roc_auc_score, average_precision_score

val_prob = model.predict_proba(X_val)[:, 1]

roc_auc = roc_auc_score(y_val, val_prob)
pr_auc = average_precision_score(y_val, val_prob)
base_rate = y_val.mean()

print(f"Validation ROC-AUC: {roc_auc:.4f}")
print(f"Validation PR-AUC: {pr_auc:.4f}")
print(f"Validation base rate: {base_rate:.4f}")

Validation ROC-AUC: 0.7702
Validation PR-AUC: 0.0558
Validation base rate: 0.0159


In [0]:
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name="logistic_regression_2015_2019"):
    
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_param("training_vintages", "2015-2019")
    mlflow.log_param("validation_vintage", "2020")
    mlflow.log_param("num_features", len(numeric_features) + len(categorical_features))
    
    mlflow.log_metric("validation_roc_auc", roc_auc)
    mlflow.log_metric("validation_pr_auc", pr_auc)
    mlflow.log_metric("validation_base_rate", base_rate)
    
    mlflow.sklearn.log_model(model, "model")

print("MLflow run logged.")

2026/09/29 19:57:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-ab4b2552-b674.cloud.databricks.com/ml/experiments/2167140964880092/models/m-94dd2409e9be42d7b81d355f26fecc60?o=7474655126366375
2026/09/29 19:58:02 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.12.0/ml/model/signatures.html for instructions on setting signature on models.


MLflow run logged.


In [0]:
from mlflow.models import infer_signature

signature = infer_signature(
    X_train,
    model.predict_proba(X_train)[:, 1]
)

input_example = X_train.head(5)

with mlflow.start_run(run_name="logistic_regression_2015_2019_final"):
    
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_param("training_vintages", "2015-2019")
    mlflow.log_param("validation_vintage", "2020")
    mlflow.log_param("num_features", len(numeric_features) + len(categorical_features))

    mlflow.log_metric("validation_roc_auc", roc_auc)
    mlflow.log_metric("validation_pr_auc", pr_auc)
    mlflow.log_metric("validation_base_rate", base_rate)

    mlflow.sklearn.log_model(
        model,
        name="model",
        signature=signature,
        input_example=input_example
    )

print("Final MLflow run logged with model signature.")

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-ab4b2552-b674.cloud.databricks.com/ml/experiments/2167140964880092/models/m-0a0429522e594ed5b94e55af12ff9650?o=7474655126366375


Final MLflow run logged with model signature.
